In [ ]:
import matplotlib.pyplot as plt
import open3d as o3d
import numpy as np
import json
import os

import pyFM.spectral as spectral
from pyFM.mesh import TriMesh
from pyFM.eval import evaluate
from pyFM.mesh import geometry

from tqdm.auto import tqdm
import meshplot as mp

def double_plot_o3d(myMesh1,myMesh2,cmap1=None,cmap2=None):
    vertlist_1 = np.asarray(myMesh1.vertices)
    facelist_1 = np.asarray(myMesh1.triangles)

    vertlist_2 = np.asarray(myMesh2.vertices)
    facelist_2 = np.asarray(myMesh2.triangles)
    
    d = mp.subplot(vertlist_1, facelist_1, c=cmap1, s=[2, 2, 0])
    mp.subplot(vertlist_2, facelist_2, c=cmap2, s=[2, 2, 1], data=d)
    
def double_plot(myMesh1,myMesh2,cmap1=None,cmap2=None):
    d = mp.subplot(myMesh1.vertlist, myMesh1.facelist, c=cmap1, s=[2, 2, 0])
    mp.subplot(myMesh2.vertlist, myMesh2.facelist, c=cmap2, s=[2, 2, 1], data=d)

def visu(vertices):
    min_coord,max_coord = np.min(vertices,axis=0,keepdims=True),np.max(vertices,axis=0,keepdims=True)
    cmap = (vertices-min_coord)/(max_coord-min_coord)
    return cmap

In [ ]:
def clean_mesh(mesh):

    mesh.remove_duplicated_vertices()
    mesh.remove_degenerate_triangles()
    mesh.remove_unreferenced_vertices()
    return mesh

def load_meshes(path, frames):
    mesh_list_o3d = [o3d.io.read_triangle_mesh(f'{path}/render_frame_{frame}.ply') for frame in tqdm(frames)]
    mesh_list = []
    for mesh in tqdm(mesh_list_o3d):

        cleanMesh = clean_mesh(mesh)

        # mesh_simplified = mesh.simplify_quadric_decimation(target_number_of_triangles=1000 * 2)  # ~2 faces per vertex
        verts, faces = np.asarray(cleanMesh.vertices), np.asarray(cleanMesh.triangles)
        
        # Create a TriMesh
        mesh_pyfm = TriMesh(verts, faces, area_normalize=True, center=True)
        mesh_pyfm.process(k=150, intrinsic=True) # why is this section so hard to process!!!!
        mesh_list.append(mesh_pyfm)

    return mesh_list

def load_gt_pcds(path, frames):
    gt_pcds = []
    for frame in tqdm(frames):
        all_pts = []

        for i in range(4):
            with open(f'{path}/ground_truth/Camera_{i+1}/frame_{frame}.json', 'r') as f:
                camera_hit_list = json.load(f)

            # Extract just the hit points
            points = [entry['hit'] for entry in camera_hit_list if entry['hit'] is not None]
            points_np = np.array(points, dtype=np.float32)
            all_pts.append(points_np)

        # combine points into PCD
        combined_pts = np.vstack(all_pts)
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(combined_pts)
        gt_pcds.append(pcd)

    return gt_pcds

path_renders = os.path.abspath('renders_small_dataset')
path_gt = os.path.abspath('renders_small_dataset')

start_frame = 1
end_frame = 10
frames = [str(i).zfill(4) for i in range(start_frame, end_frame + 1)]

mesh_list = load_meshes(path_renders, frames)
gt_pcds = load_gt_pcds(path_gt, frames)


In [ ]:
def initial_p2p_map(verts_i, verts_j):

    pcd_j = o3d.geometry.PointCloud() # create pointcloud
    pcd_j.points = o3d.utility.Vector3dVector(verts_j) # assign verts with normals to cloud

    kdtree = o3d.geometry.KDTreeFlann(pcd_j)
    
    p2p_ji = []
    for vert_i in verts_i:
        [_, idx, _] = kdtree.search_knn_vector_3d(vert_i, 1)
        p2p_ji.append(idx[0])

    p2p_ji = np.array(p2p_ji)
    return p2p_ji

In [ ]:
K = 30 # intial dimensions
maps_dict = {}
n_meshes = len(mesh_list)

for i in tqdm(range(n_meshes)):
    for j in range(n_meshes):
        if i == j: 
            continue # skip self-maps

        mesh_i, mesh_j = mesh_list[i], mesh_list[j]

        verts_i, verts_j = np.asarray(mesh_i.vertices), np.asarray(mesh_j.vertices)
        p2p_ji = initial_p2p_map(verts_j, verts_i)

        # Convert to functional map
        FM_ij = spectral.mesh_p2p_to_FM(p2p_ji, mesh_i, mesh_j, dims=K)
    
        # Populate the dictionary
        maps_dict[(i, j)] = FM_ij

In [ ]:
from pyFM.FMN import FMN

# Build the network
fmn_model = FMN(mesh_list, maps_dict.copy())
fmn_model.compute_CCLB(m=20)

In [ ]:
# Run Consistent Zoomout
fmn_model.zoomout_refine(nit=25, step=2, subsample=3000, isometric=True, weight_type='icsm',
                    M_init=None, cclb_ratio=.9, n_jobs=1, equals_id=False,
                    verbose=True)

In [ ]:
# To Save or Load Model
import pickle as p

#save
with open("../../fmn_model_10.pkl", "wb") as f:
    p.dump(fmn_model, f)

# load
# with open("../../fmn_model_MESHES_NEW.pkl", "rb") as f:
#     fmn_model = p.load(f)

In [ ]:
# For Display
ind_1 = 2
ind_2 = 7

mesh1, mesh2 = mesh_list[ind_1], mesh_list[ind_2]

new_maps_dict = fmn_model.maps
n_map = new_maps_dict[(ind_1, ind_2)]

print(f"new map shape: {n_map.shape}")

plt.matshow(n_map)
plt.title(f"C mat before ZoomOut - mesh {ind_1} -> mesh {ind_2}")
plt.show()

# mesh1_o3d, mesh2_o3d = mesh_list_o3d[ind_1], mesh_list_o3d[ind_2]
# verts_o3d = np.asarray(mesh1_o3d.vertices)

p2p_21 = spectral.convert.mesh_FM_to_p2p(n_map, mesh1, mesh2) # need to look at this function properly

cmap_1 = visu(mesh1.vertlist)
cmap_2 = cmap_1[p2p_21]
double_plot(mesh1, mesh2, cmap_1, cmap_2)

print(mesh1.vertlist.shape)
print(mesh2.vertlist.shape)
print(p2p_21.shape)

# For Ground Truth

In [ ]:
accuracy = []
for i in tqdm(range(len(mesh_list))):
    for j in range(n_meshes):
        if i == j: 
            continue # skip self-maps

        # ground truth map
        pcd_i = gt_pcds[i], pcd_j = gt_pcds[j]
        p2p_gt = initial_p2p_map(pcd_j, pcd_i)

        # output
        mesh1, mesh2 = mesh_list[i], mesh_list[j]
        D_geod_i = geometry.geodesic_distmat_dijkstra(mesh1.vertlist, mesh1.facelist)

        new_maps_dict = fmn_model.maps
        n_map = new_maps_dict[(i, j)]
        p2p_ji = spectral.convert.mesh_FM_to_p2p(n_map, mesh1, mesh2)

        acc = evaluate.accuracy(p2p_ji, p2p_gt, D_geod_i)
        accuracy.append(acc)

# plot?
print("Average geodesic accuracy across all pairs:", np.mean(accuracy))





        
